# Multimodal RAG for pdfs, videos, images, webpages and text

In [ ]:
# Dependencies
!pip install -q google-generativeai langchain langchain-google-genai \
    langchain-community langchain-text-splitters langchain-core chromadb
!pip install pypdf
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate decord
!pip install gdown

In [ ]:
# Imports

import os
import re
import torch
import numpy as np
from PIL import Image
from decord import VideoReader, cpu
from transformers import AutoProcessor, VideoLlavaForConditionalGeneration

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader

import gdown

In [ ]:
# Loading Video-LLaVA model for summarizing videos and images

model_id = "LanguageBind/Video-LLaVA-7B-hf"

processor = AutoProcessor.from_pretrained(model_id)
model = VideoLlavaForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

In [ ]:
# Downloading and extracting sample documents
file_id = "1Y_PdwASY4EcnLLlO9UmuoXgR1EfVAdRy"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "documents.zip",
    quiet=False
)

!unzip -q documents.zip -d /content

In [ ]:
# Setting up directories in colab
# Run this if you did not run the above cell and get sampler documents
# Manually place the documents in the relevant folder

content_dir = "/content/documents"
pdf_dir = os.path.join(content_dir, "pdf")
text_dir = os.path.join(content_dir, "text")
img_dir = os.path.join(content_dir, "image")
vid_dir = os.path.join(content_dir, "video")

for d in [content_dir, pdf_dir, text_dir, img_dir, vid_dir]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Extracting list of paths of uploaded documents

pdf_list = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith(".pdf")]
text_list = [os.path.join(text_dir, f) for f in os.listdir(text_dir) if f.endswith(".txt")]
image_list = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith((".png",".jpg",".jpeg"))]
video_list = [os.path.join(vid_dir, f) for f in os.listdir(vid_dir) if f.lower().endswith((".mp4",".mov",".avi"))]

# List of urls to be used as document (Add other urls to this list if needed)
url_list = ["https://www.rfc-editor.org/rfc/rfc1149"]


In [ ]:
# Cleaning webpage documents
def clean_webpage_content(text: str) -> str:
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"RFC\s+\d+.*?\n", "", text)
    text = re.sub(r"Page\s+\d+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


In [ ]:
# Summarizing videos and images

def summarize_video(video_path, chunk_size_seconds=5, frames_per_chunk=6):
    """
    Summarizes given video using Video-LLaVA model. This summary is used as docuemnt later for checking retrieval.
    Each video is broken down into chunks of chunk_size_seconds and each chunk extracts only frames_per_chunk number of frames.
    """

    prompt = "USER: <video>Describe step by step what is happening in this video in detail with all objects and colors. ASSISTANT:"

    vr = VideoReader(video_path, ctx=cpu())
    fps = float(vr.get_avg_fps())
    duration = len(vr) / fps

    documents = []

    for start_time in np.arange(0, duration, chunk_size_seconds):

        # Processing each chunk
        end_time = min(start_time + chunk_size_seconds, duration)

        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)

        if end_frame <= start_frame:
            continue

        indices = np.linspace(start_frame, end_frame - 1, frames_per_chunk).astype(int)
        frames = vr.get_batch(indices).asnumpy()

        inputs = processor(text=prompt, videos=frames, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=180,
                do_sample=False
            )

        # Removing initial prompt from summary :
        generated = output[0, inputs["input_ids"].shape[1]:]
        decoded = processor.decode(generated, skip_special_tokens=True).strip()

        if "ASSISTANT:" in decoded:
            decoded = decoded.split("ASSISTANT:")[-1].strip()

        doc_text = f"""
        [Video Segment]
        Source file: {os.path.basename(video_path)}
        Segment: {start_time:.1f}s - {end_time:.1f}s

        Description:
        {decoded}
        """

        documents.append(
            Document(
                page_content=doc_text.strip(),
                metadata={
                    "source_type": "video",
                    "source": video_path,
                    "filename": os.path.basename(video_path),
                    "start_time": float(start_time),
                    "end_time": float(end_time)
                }
            )
        )

    return documents

def summarize_image(image_path):
    """
    Summarizes given image using Video-LLaVA model. This summary is used as docuemnt later for checking retrieval.
    """

    prompt = "USER: <image>Describe this image in detail with all objects, colors and steps (if it is a flow diagram). ASSISTANT:"

    image = Image.open(image_path).convert("RGB")

    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

    # These hyperparams work better with images (need to test further)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15
        )

    # Removing initial prompt from summary :
    generated = output[0, inputs["input_ids"].shape[1]:]
    decoded = processor.decode(generated, skip_special_tokens=True).strip()

    if "ASSISTANT:" in decoded:
        decoded = decoded.split("ASSISTANT:")[-1].strip()

    doc_text = f"""
    [Image Document]
    Source file: {os.path.basename(image_path)}

    Description:
    {decoded}
    """

    return Document(
        page_content=doc_text.strip(),
        metadata={
            "source_type": "image",
            "source": image_path,
            "filename": os.path.basename(image_path)
        }
    )


In [ ]:
# Load all pdfs, texts and webpages to documents
loaders = []

for p in pdf_list:
    loaders.append(PyPDFLoader(p))

for t in text_list:
    loaders.append(TextLoader(t))

for u in url_list:
    loaders.append(WebBaseLoader(u))

all_docs = []

for loader in loaders:
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_type"] = loader.__class__.__name__
        if isinstance(loader, WebBaseLoader):
            doc.page_content = clean_webpage_content(doc.page_content)
    all_docs.extend(docs)

In [ ]:
# Load all images to document list

print("Processing images :")
for img_path in image_list:
    try:
        img_doc = summarize_image(img_path)
        all_docs.append(img_doc)
        print(f"Processed image: {img_path}")
    except Exception as e:
        # exception if any image fails to load
        print(f"Failed image {img_path}: {e}")

# Load all videos to document list

print("Processing videos :")
for vid_path in video_list:
    try:
        video_docs = summarize_video(vid_path)
        all_docs.extend(video_docs)
        print(f"Processed video: {vid_path}")
    except Exception as e:
        # exception if any video fails to load
        print(f"Failed video {vid_path}: {e}")


In [ ]:
# Use your own Gemini API Key here
from google.colab import userdata

gemini_api_key_name = "<PUT-YOUR-KEY-HERE>"
GOOGLE_API_KEY = userdata.get(gemini_api_key_name)
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

In [ ]:
# Chunking all documents

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
docs = splitter.split_documents(all_docs)

In [ ]:
# Getting Embeddings for each chunk
# and creating ANN index using HNSW (Hierarchical Navigable Small World graphs)
# for initial retrieval of top-k (here 4) relevant documents

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
# Initializing LLM and setting up prompt

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer ONLY using the provided context. "
     "If the answer is not in the context, say 'Not found in documents.'\n\n"
     "Context:\n{context}"
    ),
    ("human", "{input}")
])

# Settting up rag chain with documents and LLM
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

Test 1 : Success

In [ ]:
result = rag_chain.invoke({"input": "What is the color of the purse carried by the woman in the white dress ?"})

In [ ]:
print("Answer:", result["answer"])

Answer: The purse is silver in color.


In [ ]:
for i, doc in enumerate(result["context"], 1):
    print(f"Document {i}:")
    print(doc.page_content)
    print("-" * 60)
print(f"Quesion : {result['input']}")
print("-" * 60)
print("Answer:", result["answer"])

Document 1:
Description:
        In the video, a woman is walking down a runway while carrying a large purse. She is wearing a white dress and is the main focus of the scene. The purse is silver in color and is held in her right hand. The woman is walking on a stage, and there are other people in the background, but they are not the main focus of the video. The stage is well-lit, and the woman's dress is flowing, giving her a graceful appearance. The video captures the moment of the woman's walk down the runway, showcasing her fashionable attire and the purse she is carrying.
------------------------------------------------------------
Document 2:
Description:
        In the video, a woman is walking down a runway while holding a purse. She is wearing a white dress and is the main focus of the scene. There are several other people in the background, some of whom are also wearing white dresses. The woman is walking in front of a large crowd, and the audience seems to be watching her as 

Test 2 : Success

In [ ]:
result = rag_chain.invoke({"input": "Explain the email process flowchart ?"})

In [ ]:
print("Answer:", result["answer"])

Answer: The email process flowchart is drawn on a white piece of paper and maps the workflow from start to finish through several stages:

*   **Request for Data:** Located at the top left corner.
*   **Data Input:** Located below the "Request for Data" box.
*   **Email Setup** and **Mail Send:** Two boxes following underneath each other.
*   **Email Received**, **Read by Recipient**, **Response Sent**, and **Response Read:** Four boxes arranged in sequence further down.

The chart also maps stages involved in sending emails, such as "To" recipients, "Cc" senders, and "Bcc" addresses.


In [ ]:
for i, doc in enumerate(result["context"], 1):
    print(f"Document {i}:")
    print(doc.page_content)
    print("-" * 60)
print(f"Quesion : {result['input']}")
print("-" * 60)
print("Answer:", result["answer"])

Document 1:
In total, there are six boxes representing various stages, including one labeled "Request for Data" at the top left corner of the paper. Below that box, another box appears titled "Data Input". Then, two more boxes follow underneath each other – "Email Setup" and "Mail Send." Moving down further, we find four more boxes arranged in sequence – "Email Received", "Read by Recipient", "Response Sent," and finally, "Response Read". Each step of the process is carefully defined within these boxes.
------------------------------------------------------------
Document 2:
[Image Document]
    Source file: img5.png

    Description:
    The image shows a white piece of paper on which an email workflow is neatly drawn out. Mapping the process from start to finish, the chart consists of different stages involved in sending emails, such as "To" recipients, "Cc" senders, and "Bcc" addresses.
------------------------------------------------------------
Document 3:
Upon receipt, the duct t

Test 3 : Success

In [ ]:
result = rag_chain.invoke({"input": "What are big hurdles for quantum computers ?"})

In [ ]:
print("Answer:", result["answer"])

Answer: Quantum systems suffer from decoherence, and noise and error rates are major obstacles. Scalability remains an engineering challenge. Additionally, many quantum computers use superconducting circuits which must be cooled to near absolute zero temperatures.


In [ ]:
for i, doc in enumerate(result["context"], 1):
    print(f"Document {i}:")
    print(doc.page_content)
    print("-" * 60)
print(f"Quesion : {result['input']}")
print("-" * 60)
print("Answer:", result["answer"])

Document 1:
Quantum computers use qubits instead of classical bits.
A qubit can exist in a superposition of 0 and 1 simultaneously.
This allows certain problems to be solved more efficiently than classical systems.
Shor's algorithm is used for integer factorization.
Grover's algorithm speeds up search problems.
Quantum advantage refers to outperforming classical computers.
Quantum systems suffer from decoherence.
Noise and error rates are major obstacles.
Scalability remains an engineering challenge.
Classical computers use binary bits (0 or 1).
They rely on transistors and deterministic logic gates.
------------------------------------------------------------
Document 2:
Entanglement is a uniquely quantum phenomenon where particles share correlated states.
Quantum error correction is essential for building reliable quantum computers.
Artificial intelligence and quantum computing may intersect in optimization tasks.
Hybrid quantum-classical algorithms are being actively researched.
The

Test 4 (Hallucination test) : Success

In [ ]:
result = rag_chain.invoke({"input": "What are topological qubits and how do they reduce decoherence?"})

In [ ]:
print("Answer:", result["answer"])

Answer: Not found in documents.


In [ ]:
for i, doc in enumerate(result["context"], 1):
    print(f"Document {i}:")
    print(doc.page_content)
    print("-" * 60)
print(f"Quesion : {result['input']}")
print("-" * 60)
print("Answer:", result["answer"])

Document 1:
Quantum computers use qubits instead of classical bits.
A qubit can exist in a superposition of 0 and 1 simultaneously.
This allows certain problems to be solved more efficiently than classical systems.
Shor's algorithm is used for integer factorization.
Grover's algorithm speeds up search problems.
Quantum advantage refers to outperforming classical computers.
Quantum systems suffer from decoherence.
Noise and error rates are major obstacles.
Scalability remains an engineering challenge.
Classical computers use binary bits (0 or 1).
They rely on transistors and deterministic logic gates.
------------------------------------------------------------
Document 2:
They rely on transistors and deterministic logic gates.
Many quantum computers use superconducting circuits.
These must be cooled to near absolute zero temperatures.
Tomatoes grow best in well-drained soil.
Regular watering improves yield.
Stock prices fluctuate due to supply and demand.
Cryptography is used in secure